In [3]:
import pickle
import pandas as pd
import numpy as np
import os

# Imports for plotting and statistical tests
import matplotlib
matplotlib.use('agg') # Use non-interactive backend for saving files
import matplotlib.pyplot as plt
import operator
import math
from scipy.stats import wilcoxon, friedmanchisquare
import networkx

# Imports for metrics
from sklearn.metrics import precision_score, recall_score, f1_score, balanced_accuracy_score, accuracy_score

# =============================================================================
# 1. SETUP AND CONFIGURATION
# =============================================================================
from d2c.benchmark.utils import draw_cd_diagram

THRESHOLD = 0.55

OUTPUT_PATH_CD = "CD_PLOTS/"
os.makedirs(OUTPUT_PATH_CD, exist_ok=True)

ORDER_SAVING_RESULTS = [
    'var', 'varlingam', 'pcmci', 'mvgc', 'pcmci_gpdc',
    'granger', 'dynotears', 'td2c'
]

DISPLAY_NAMES = {
    'var': 'VAR', 'varlingam': 'VARLINGAM', 'pcmci': 'PCMCI', 'mvgc': 'MVGC',
    'pcmci_gpdc': 'PCMCI-GPDC', 'granger': 'GRANGER', 'dynotears': 'DYNOTEARS', 'td2c': 'TD2C'
}

plt.rcParams['font.family'] = 'DejaVu Sans'  # Use available font


datasets_name = ['TEST']
for dataset_name in datasets_name:
    file_path = f'data/causal_dfs/causal_dfs_{dataset_name}.pkl'
    print(f"--- Creating Original CD Plot for {dataset_name} ---")

    # --- Load Data ---
    with open(file_path, 'rb') as f:
        loaded_data = pickle.load(f)
    method_results_tuple = loaded_data[:-1]
    true_causal_dfs_dict = loaded_data[-1]

    # --- Calculate Precision scores for each run using pre-computed predictions ---
    run_ids = sorted(true_causal_dfs_dict.keys())

    metrics_function_dict = {'precision': precision_score,
                                'recall': recall_score,
                                'f1': f1_score,
                                'balanced_accuracy': balanced_accuracy_score,
                                'accuracy': accuracy_score}

    for metric, metric_function in metrics_function_dict.items():
        per_run_scores = []
        for run_id in run_ids:
            y_true_run = true_causal_dfs_dict[run_id]['is_causal'].astype(int)

            for i, method_dfs_dict in enumerate(method_results_tuple):
                internal_name = ORDER_SAVING_RESULTS[i]
                
                if method_dfs_dict is None or run_id not in method_dfs_dict:
                    continue

                if internal_name == 'td2c':
                    y_proba_run = method_dfs_dict[run_id]['probability'].astype(float)
                    y_pred_run = (y_proba_run > THRESHOLD).astype(int)
                else:
                    y_pred_run = method_dfs_dict[run_id]['is_causal'].astype(int)
                
                if metric == 'precision' or metric == 'recall' or metric == 'f1':
                    score = metric_function(y_true_run, y_pred_run, zero_division=np.nan)
                elif metric == 'balanced_accuracy' or metric == 'accuracy':
                    score = metric_function(y_true_run, y_pred_run)
                
                per_run_scores.append({
                    'Model': DISPLAY_NAMES[internal_name],
                    'dataset_name': f"{dataset_name}_{run_id}",
                    'Score': score, # The score for this run
                })
        
        # --- Create a DataFrame suitable for the CD plot function ---
        scores_df = pd.DataFrame(per_run_scores)
        
        # --- Generate the CD plot for Precision ---
        print("\nGenerating Precision CD plot...")
        output_path = f"{OUTPUT_PATH_CD}/cd_{dataset_name}_{metric}.png"
        draw_cd_diagram(
            path=output_path,
            df_perf=scores_df,
        )

    print("\nScript finished successfully.")

--- Creating Original CD Plot for TEST ---

Generating Precision CD plot...
['VAR' 'VARLINGAM' 'PCMCI' 'MVGC' 'PCMCI-GPDC' 'GRANGER' 'DYNOTEARS'
 'TD2C']
        Model  count
0   DYNOTEARS   1080
1     GRANGER   1080
2        MVGC   1080
3       PCMCI   1080
4  PCMCI-GPDC   1080
5        TD2C   1080
6         VAR   1080
7   VARLINGAM   1080
[[       nan        nan        nan ...        nan        nan        nan]
 [0.14285714 0.23529412 0.         ... 0.         0.         0.11764706]
 [0.51724138 0.62962963 1.         ... 1.         1.         0.6       ]
 ...
 [0.68421053 0.57894737 0.8        ... 0.83333333 0.7        0.57142857]
 [0.5        0.8        1.         ... 0.         0.         0.57142857]
 [0.47826087 0.40540541 0.68421053 ... 0.83333333 0.77777778 0.375     ]]
DYNOTEARS      42.0
GRANGER         4.0
MVGC           74.0
PCMCI          64.0
PCMCI-GPDC    100.0
TD2C          427.0
VAR           106.0
VARLINGAM      17.0
dtype: float64
GRANGER       6.481481
VAR           4

In [11]:
import pickle
import pandas as pd
import numpy as np
import os
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, balanced_accuracy_score, accuracy_score
# Assuming your d2c.benchmark.cd_plot is in the path
from d2c.benchmark.cd_plot import draw_cd_diagram

# =============================================================================
# 1. SETUP AND CONFIGURATION
# =============================================================================

ORDER_SAVING_RESULTS = [
    'var', 'varlingam', 'pcmci', 'mvgc', 'pcmci_gpdc',
    'granger', 'dynotears', 'td2c'
]

DISPLAY_NAMES = {
    'var': 'VAR', 'varlingam': 'VARLINGAM', 'pcmci': 'PCMCI', 'mvgc': 'MVGC',
    'pcmci_gpdc': 'PCMCI-GPDC', 'granger': 'GRANGER', 'dynotears': 'DYNOTEARS', 'td2c': 'TD2C'
}

DATASET_TYPES = ['NETSIM_5', 'NETSIM_10', 'DREAM3_10', 'DREAM3_50', 'TEST']

plt.rcParams['font.family'] = 'DejaVu Sans'

# =============================================================================
# 2. MAIN SCRIPT
# =============================================================================

def main():
    print("--- Running Robust Statistical Comparison ---")
    print("Method: Averaging performance over runs within each dataset type, then comparing types.\n")

    metrics_to_run = {
        'precision': precision_score,
        'recall': recall_score,
        'f1_score': f1_score,
        'balanced_accuracy': balanced_accuracy_score,
        'accuracy': accuracy_score
    }

    # This will hold the final aggregated scores for all metrics
    # e.g., {'precision': df_for_precision_plot, 'recall': df_for_recall_plot}
    final_dfs_for_plotting = {metric_name: [] for metric_name in metrics_to_run.keys()}

    # --- Step 1: Calculate average performance for each model on each dataset type ---
    for dataset_type in DATASET_TYPES:
        file_path = f'data/causal_dfs_{dataset_type}.pkl'
        if not os.path.exists(file_path):
            print(f"Skipping {dataset_type}: File not found.")
            continue
        
        print(f"Processing {dataset_type}...")
        
        with open(file_path, 'rb') as f:
            loaded_data = pickle.load(f)
        method_results_tuple = loaded_data[:-1]
        true_causal_dfs_dict = loaded_data[-1]

        # Find common runs for this dataset type to ensure fair averaging
        all_dicts = [d for d in method_results_tuple if d is not None] + [true_causal_dfs_dict]
        if not all_dicts: continue
        common_keys = set(all_dicts[0].keys())
        for d in all_dicts[1:]:
            common_keys.intersection_update(d.keys())

        # For each metric, calculate the average score
        for metric_name, metric_function in metrics_to_run.items():
            for i, method_dfs_dict in enumerate(method_results_tuple):
                internal_name = ORDER_SAVING_RESULTS[i]
                if method_dfs_dict is None:
                    continue

                run_scores = []
                for run_id in sorted(list(common_keys)):
                    if run_id not in method_dfs_dict: continue
                    
                    y_true_run = true_causal_dfs_dict[run_id]['is_causal'].astype(int)
                    pred_df_run = method_dfs_dict[run_id]
                    y_pred_run = pred_df_run['is_causal'].astype(int)
                    
                    # Calculate score for this single run
                    if metric_name in ['precision', 'recall', 'f1_score']:
                        score = metric_function(y_true_run, y_pred_run, zero_division=0.0)
                    else:
                        score = metric_function(y_true_run, y_pred_run)
                    run_scores.append(score)
                
                # If there are scores, calculate the mean and store it
                if run_scores:
                    avg_score = np.mean(run_scores)
                    final_dfs_for_plotting[metric_name].append({
                        'Model': DISPLAY_NAMES[internal_name],
                        'dataset_name': dataset_type,  # The "subject" is now the dataset type
                        'Score': avg_score
                    })

    # --- Step 2: Generate a CD plot for each metric using the aggregated scores ---
    print("\n--- Generating Final CD Plots ---")
    for metric_name, data_list in final_dfs_for_plotting.items():
        if not data_list:
            print(f"No data to plot for metric: {metric_name}")
            continue
            
        # Create the final DataFrame for this metric
        scores_df = pd.DataFrame(data_list)
        
        # Handle cases where a model failed on an entire dataset type
        scores_df['Score'].fillna(0.0, inplace=True)
        
        print(f"\nGenerating CD plot for: {metric_name}...")
        
        # This DataFrame is now small and robust (e.g., 8 models x 5 datasets)
        # It's the correct input for the statistical tests.
        output_path = f"figures/cd_robust_{metric_name}.png"
        draw_cd_diagram(
            path=output_path,
            df_perf=scores_df
            # You can add a title if your `draw_cd_diagram` supports it
            # title=f"CD Diagram for {metric_name.replace('_', ' ').title()}"
        )

    print("\nScript finished successfully.")

if __name__ == '__main__':
    main()

--- Running Robust Statistical Comparison ---
Method: Averaging performance over runs within each dataset type, then comparing types.

Processing NETSIM_5...
Processing NETSIM_10...
Processing DREAM3_10...
Processing DREAM3_50...
Processing TEST...

--- Generating Final CD Plots ---

Generating CD plot for: precision...
['VAR' 'VARLINGAM' 'PCMCI' 'MVGC' 'PCMCI-GPDC' 'GRANGER' 'DYNOTEARS'
 'TD2C']
        Model  count
0   DYNOTEARS      5
1     GRANGER      5
2        MVGC      5
3       PCMCI      5
4  PCMCI-GPDC      3
5        TD2C      5
6         VAR      5
7   VARLINGAM      4
[[0.23915072 0.0718026  0.10298082 0.20850769 0.18049604]
 [0.08142216 0.01492691 0.07486515 0.10227196 0.06736236]
 [0.10303221 0.08694043 0.27472686 0.35835553 0.4849838 ]
 [0.36603975 0.26228096 0.31957953 0.34503954 0.49759595]
 [0.81111111 0.85363147 0.85195953 0.91690041 0.75404281]
 [0.0973545  0.062366   0.18010259 0.14666268 0.33513033]]
DYNOTEARS    0.0
GRANGER      0.0
MVGC         0.0
PCMCI      

In [8]:
import pickle
import pandas as pd
import numpy as np
import os
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, balanced_accuracy_score, accuracy_score
# Assuming your d2c.benchmark.cd_plot is in the path
from d2c.benchmark.cd_plot import draw_cd_diagram, wilcoxon_holm

# =============================================================================
# 1. SETUP AND CONFIGURATION
# =============================================================================

ORDER_SAVING_RESULTS = [
    'var', 'varlingam', 'pcmci', 'mvgc', 'pcmci_gpdc',
    'granger', 'dynotears', 'td2c'
]

DISPLAY_NAMES = {
    'var': 'VAR', 'varlingam': 'VARLINGAM', 'pcmci': 'PCMCI', 'mvgc': 'MVGC',
    'pcmci_gpdc': 'PCMCI-GPDC', 'granger': 'Granger', 'dynotears': 'DYNOTEARS', 'td2c': 'TD2C'
}

DATASET_TYPES = ['NETSIM_5', 'NETSIM_10', 'DREAM3_10', 'DREAM3_50', 'TEST']

plt.rcParams['font.family'] = 'DejaVu Sans'

# =============================================================================
# 2. MAIN SCRIPT
# =============================================================================

def main():
    print("--- Running Two-Step Statistical Comparison ---")
    print("Method: 1. Rank models within each dataset type. 2. Compare ranks across types.\n")

    metrics_to_run = {
        'precision': precision_score,
        'recall': recall_score,
        'f1_score': f1_score,
        'balanced_accuracy': balanced_accuracy_score,
        'accuracy': accuracy_score
    }
    
    # This will hold the intermediate table of ranks (Models x Dataset Types)
    intermediate_ranks = {metric_name: [] for metric_name in metrics_to_run.keys()}

    # --- Step 1: Generate a table of average ranks for each dataset type ---
    for dataset_type in DATASET_TYPES:
        file_path = f'data/causal_dfs_{dataset_type}.pkl'
        if not os.path.exists(file_path):
            print(f"Skipping {dataset_type}: File not found.")
            continue
        
        print(f"Processing intermediate ranks for: {dataset_type}...")
        
        with open(file_path, 'rb') as f:
            loaded_data = pickle.load(f)
        
        # This logic remains the same: gather scores for all runs within this type
        per_run_scores_for_type = {metric_name: [] for metric_name in metrics_to_run.keys()}
        method_results_tuple = loaded_data[:-1]
        true_causal_dfs_dict = loaded_data[-1]
        run_ids = sorted(true_causal_dfs_dict.keys())
        
        for run_id in run_ids:
            y_true_run = true_causal_dfs_dict[run_id]['is_causal'].astype(int)
            for i, method_dfs_dict in enumerate(method_results_tuple):
                internal_name = ORDER_SAVING_RESULTS[i]
                if method_dfs_dict is None or run_id not in method_dfs_dict:
                    continue
                
                pred_df_run = method_dfs_dict[run_id]
                y_pred_run = pred_df_run['is_causal'].astype(int)
                
                for metric_name, metric_function in metrics_to_run.items():
                    if metric_name in ['precision', 'recall', 'f1_score']:
                        score = metric_function(y_true_run, y_pred_run, zero_division=0.0)
                    else:
                        score = metric_function(y_true_run, y_pred_run)
                    
                    per_run_scores_for_type[metric_name].append({
                        'Model': DISPLAY_NAMES[internal_name],
                        'dataset_name': f"{dataset_type}_{run_id}", 
                        'Score': score
                    })

        # Now, for each metric, calculate the average ranks for this dataset type
        for metric_name, data_list in per_run_scores_for_type.items():
            if not data_list: continue
            
            df_for_ranking = pd.DataFrame(data_list)
            
            # Use a simplified ranking method without plotting
            # The goal is just to get the average rank numbers
            classifiers = list(df_for_ranking['Model'].unique())
            max_nb_datasets = df_for_ranking.groupby('Model').size().max()
            
            rank_data = np.array(df_for_ranking['Score']).reshape(len(classifiers), max_nb_datasets)
            df_ranks = pd.DataFrame(data=rank_data, index=classifiers)
            average_ranks_for_type = df_ranks.rank(ascending=False).mean(axis=1)
            
            # Store these ranks
            for model, rank in average_ranks_for_type.items():
                intermediate_ranks[metric_name].append({
                    'Model': model,
                    'dataset_name': dataset_type, # The "subject" is now the dataset type
                    'Score': rank # The "score" is the rank
                })

    # --- Step 2: Generate the final CD plot from the table of ranks ---
    print("\n--- Generating Final CD Plots from Intermediate Ranks ---")
    for metric_name, data_list in intermediate_ranks.items():
        if not data_list:
            print(f"No rank data to plot for metric: {metric_name}")
            continue
            
        # Create the final DataFrame for this metric
        # The 'Score' column now contains the ranks from Step 1
        final_ranks_df = pd.DataFrame(data_list)
        
        print(f"\nGenerating CD plot for: {metric_name}...")
        print("Input to final statistical test (Models vs. Dataset Types):")
        print(final_ranks_df.pivot(index='Model', columns='dataset_name', values='Score'))
        
        output_path = f"figures/cd_final_two_step_{metric_name}.png"
        
        # This is the final, robust CD plot
        draw_cd_diagram(
            path=output_path,
            df_perf=final_ranks_df,
            alpha=0.1
            # title=f"CD Diagram for {metric_name.replace('_', ' ').title()}"
        )

    print("\nScript finished successfully.")

if __name__ == '__main__':
    main()

--- Running Two-Step Statistical Comparison ---
Method: 1. Rank models within each dataset type. 2. Compare ranks across types.

Processing intermediate ranks for: NETSIM_5...
Processing intermediate ranks for: NETSIM_10...
Processing intermediate ranks for: DREAM3_10...
Processing intermediate ranks for: DREAM3_50...
Processing intermediate ranks for: TEST...

--- Generating Final CD Plots from Intermediate Ranks ---

Generating CD plot for: precision...
Input to final statistical test (Models vs. Dataset Types):
dataset_name  DREAM3_10  DREAM3_50  NETSIM_10  NETSIM_5      TEST
Model                                                            
DYNOTEARS           4.4        3.4      4.184  4.919524  4.759722
Granger             4.2        3.6      4.148  4.747143  3.647222
MVGC                5.2        3.2      4.228  4.407619  5.044907
PCMCI               3.8        3.6      3.460  4.234286  4.291204
PCMCI-GPDC          4.8        NaN        NaN  4.298095  4.923148
TD2C              

NameError: name 'exit' is not defined